# 📡 TelecomX — Análise de Evasão de Clientes (Churn)

---

| | |
|---|---|
| **Empresa** | TelecomX (fictícia) |
| **Projeto** | Churn de Clientes |
| **Processo** | ETL + EDA (Análise Exploratória de Dados) |
| **Linguagem** | Python 3 |
| **Ambiente** | Google Colab |
| **Bibliotecas** | Pandas · NumPy · Matplotlib · Seaborn |

---

> **Contexto:** A TelecomX enfrenta um alto índice de cancelamento de clientes.
> Este notebook realiza a coleta, tratamento e análise dos dados para identificar
> os principais fatores de evasão e fornecer insumos para modelos preditivos.


---
# 📌 Seção 1 — Extração (E - Extract)

## 1.1 Importação das Bibliotecas

In [ ]:
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais globais
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✅ Bibliotecas importadas com sucesso!")


## 1.2 Importação dos Dados via API

Os dados estão disponíveis publicamente no GitHub da Telecom X em formato JSON com estrutura aninhada.
Utilizamos `requests` para buscar os dados diretamente da URL da API.


In [ ]:
# URL da API (raw do GitHub)
API_URL = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"

response = requests.get(API_URL)

if response.status_code == 200:
    raw_data = response.json()
    print(f"✅ Dados carregados com sucesso!")
    print(f"   Status HTTP  : {response.status_code}")
    print(f"   Total de registros : {len(raw_data)}")
else:
    print(f"❌ Erro ao carregar dados. Status: {response.status_code}")


## 1.3 Visualizando a Estrutura do JSON

In [ ]:
# Inspecionando a estrutura de um registro para entender o schema aninhado
print("Estrutura de um registro JSON:")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))


## 1.4 Normalização do JSON → DataFrame

In [ ]:
# O JSON possui objetos aninhados: customer, phone, internet, account
# Precisamos "achatar" (flatten) para um DataFrame tabular

records = []
for d in raw_data:
    row = {}
    # Nível raiz
    row['customerID']       = d.get('customerID')
    row['Churn']            = d.get('Churn')

    # Sub-objeto: customer
    cust = d.get('customer', {})
    row['gender']           = cust.get('gender')
    row['SeniorCitizen']    = cust.get('SeniorCitizen')
    row['Partner']          = cust.get('Partner')
    row['Dependents']       = cust.get('Dependents')
    row['tenure']           = cust.get('tenure')

    # Sub-objeto: phone
    phone = d.get('phone', {})
    row['PhoneService']     = phone.get('PhoneService')
    row['MultipleLines']    = phone.get('MultipleLines')

    # Sub-objeto: internet
    internet = d.get('internet', {})
    row['InternetService']  = internet.get('InternetService')
    row['OnlineSecurity']   = internet.get('OnlineSecurity')
    row['OnlineBackup']     = internet.get('OnlineBackup')
    row['DeviceProtection'] = internet.get('DeviceProtection')
    row['TechSupport']      = internet.get('TechSupport')
    row['StreamingTV']      = internet.get('StreamingTV')
    row['StreamingMovies']  = internet.get('StreamingMovies')

    # Sub-objeto: account
    account = d.get('account', {})
    row['Contract']         = account.get('Contract')
    row['PaperlessBilling'] = account.get('PaperlessBilling')
    row['PaymentMethod']    = account.get('PaymentMethod')
    charges = account.get('Charges', {})
    row['Charges_Monthly']  = charges.get('Monthly')
    row['Charges_Total']    = charges.get('Total')

    records.append(row)

df_raw = pd.DataFrame(records)

print(f"✅ DataFrame criado: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
df_raw.head(3)


---
# 🔧 Seção 2 — Transformação (T - Transform)

## 2.1 Conhecendo o Dataset

In [ ]:
print("=" * 55)
print("VISÃO GERAL DO DATASET")
print("=" * 55)
print(f"\nDimensões : {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
print("\n--- Tipos de dados (dtypes) ---")
print(df_raw.dtypes)
print("\n--- Informações gerais ---")
df_raw.info()


In [ ]:
# Primeiras e últimas linhas
print("Primeiras 5 linhas:")
display(df_raw.head())
print("\nÚltimas 5 linhas:")
display(df_raw.tail())


## 2.2 Verificando Inconsistências nos Dados

In [ ]:
print("=" * 55)
print("VERIFICAÇÃO DE INCONSISTÊNCIAS")
print("=" * 55)

# 1. Valores nulos
print("\n[1] Valores nulos por coluna:")
nulls = df_raw.isnull().sum()
null_pct = (nulls / len(df_raw) * 100).round(2)
null_df = pd.DataFrame({'Nulos': nulls, '% Total': null_pct})
print(null_df[null_df['Nulos'] > 0].to_string() if null_df['Nulos'].sum() > 0 else "  Nenhum valor nulo encontrado.")

# 2. Linhas duplicadas
dupes = df_raw.duplicated().sum()
print(f"\n[2] Linhas duplicadas: {dupes}")

# 3. Valores únicos em Churn
print(f"\n[3] Valores únicos em 'Churn': {df_raw['Churn'].unique()}")
print(f"     Registros com Churn vazio: {(df_raw['Churn'].str.strip() == '').sum()}")

# 4. Tipo de Charges_Total
print(f"\n[4] Tipo de 'Charges_Total': {df_raw['Charges_Total'].dtype}")
print(f"     Amostra: {df_raw['Charges_Total'].head(5).tolist()}")

# 5. Categorias únicas das principais variáveis categóricas
cat_cols = ['Contract','PaymentMethod','InternetService','gender']
print("\n[5] Categorias únicas das variáveis principais:")
for col in cat_cols:
    print(f"     {col}: {df_raw[col].unique()}")


## 2.3 Tratando as Inconsistências

In [ ]:
# Trabalharemos em uma cópia para preservar o raw
df = df_raw.copy()

# ── [1] Remover registros sem rótulo de Churn ────────────────
antes = len(df)
df = df[df['Churn'].str.strip() != ''].copy()
depois = len(df)
print(f"[1] Registros sem Churn removidos: {antes - depois} → {depois} registros mantidos")

# ── [2] Converter Charges_Total para numérico ────────────────
df['Charges_Total'] = pd.to_numeric(df['Charges_Total'], errors='coerce')
print(f"[2] 'Charges_Total' convertido para float.")

# ── [3] Corrigir nulos em Charges_Total ─────────────────────
nulos_total = df['Charges_Total'].isna().sum()
df.loc[df['Charges_Total'].isna(), 'Charges_Total'] = (
    df.loc[df['Charges_Total'].isna(), 'Charges_Monthly'] *
    df.loc[df['Charges_Total'].isna(), 'tenure']
)
print(f"[3] Nulos em 'Charges_Total' corrigidos: {nulos_total}")

# ── [4] Verificação final ────────────────────────────────────
print(f"\n✅ Dataset limpo: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"   Nulos restantes: {df.isnull().sum().sum()}")


## 2.4 Coluna de Contas Diárias

In [ ]:
# Charges_Daily = valor mensal / 30 dias
df['Contas_Diarias'] = (df['Charges_Monthly'] / 30).round(4)

print("✅ Coluna 'Contas_Diarias' criada.")
df[['customerID', 'Charges_Monthly', 'Contas_Diarias']].head()


## 2.5 Padronização e Transformação de Dados

In [ ]:
# ── Mapeamento de variáveis binárias Yes/No → 1/0 ────────────
binarias = [
    'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

map_bin = {'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}
for col in binarias:
    df[col] = df[col].map(map_bin).fillna(df[col])

# ── Churn: Yes/No → 1/0 ─────────────────────────────────────
df['Churn_num'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── SeniorCitizen: label legível ─────────────────────────────
df['SeniorCitizen_label'] = df['SeniorCitizen'].map({0: 'Não', 1: 'Sim'})

# ── Faixa de cobrança mensal ─────────────────────────────────
df['Faixa_Mensal'] = pd.cut(
    df['Charges_Monthly'],
    bins=[0, 30, 60, 90, 120],
    labels=['R$0–30', 'R$30–60', 'R$60–90', 'R$90–120']
)

# ── Tradução dos nomes das colunas para pt-BR (referência) ───
colunas_ptbr = {
    'customerID': 'ID_Cliente', 'Churn': 'Evasao', 'gender': 'Genero',
    'SeniorCitizen': 'Idoso', 'Partner': 'Parceiro', 'Dependents': 'Dependentes',
    'tenure': 'Meses_Contrato', 'PhoneService': 'Servico_Telefone',
    'MultipleLines': 'Multiplas_Linhas', 'InternetService': 'Servico_Internet',
    'OnlineSecurity': 'Seguranca_Online', 'OnlineBackup': 'Backup_Online',
    'DeviceProtection': 'Protecao_Dispositivo', 'TechSupport': 'Suporte_Tecnico',
    'StreamingTV': 'Streaming_TV', 'StreamingMovies': 'Streaming_Filmes',
    'Contract': 'Tipo_Contrato', 'PaperlessBilling': 'Fatura_Digital',
    'PaymentMethod': 'Metodo_Pagamento',
    'Charges_Monthly': 'Cobranca_Mensal', 'Charges_Total': 'Cobranca_Total'
}

df_ptbr = df.rename(columns=colunas_ptbr)

print("✅ Padronização concluída.")
print(f"   Colunas no dataset: {list(df.columns)}")
df.head(3)


---
# 📊 Seção 3 — Carga e Análise (L - Load & Analysis)

## 3.1 Análise Descritiva

In [ ]:
print("=" * 55)
print("ANÁLISE DESCRITIVA — VARIÁVEIS NUMÉRICAS")
print("=" * 55)
display(df[['tenure', 'Charges_Monthly', 'Charges_Total', 'Contas_Diarias']].describe().round(2))


In [ ]:
print("=" * 55)
print("ANÁLISE DESCRITIVA — VARIÁVEIS CATEGÓRICAS")
print("=" * 55)
cat_vars = ['Churn', 'gender', 'SeniorCitizen_label', 'Contract',
            'PaymentMethod', 'InternetService']
for col in cat_vars:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())


## 3.2 Distribuição da Evasão (Churn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

churn_counts = df['Churn'].value_counts()
colors_churn = ['#2ecc71', '#e74c3c']

# Barras
bars = axes[0].bar(
    ['Não Evadiu', 'Evadiu'], churn_counts.values,
    color=colors_churn, edgecolor='white', width=0.5
)
axes[0].set_title('Contagem de Clientes por Churn', fontweight='bold')
axes[0].set_ylabel('Quantidade de Clientes')
axes[0].set_ylim(0, churn_counts.max() * 1.15)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 60,
                 f'{h:,}', ha='center', fontweight='bold', fontsize=12)

# Pizza
axes[1].pie(
    churn_counts.values,
    labels=['Não Evadiu', 'Evadiu'],
    colors=colors_churn,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
axes[1].set_title('Proporção de Churn', fontweight='bold')

plt.suptitle('Distribuição da Evasão de Clientes — TelecomX',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

taxa = df['Churn_num'].mean() * 100
print(f"\n📊 Taxa geral de evasão: {taxa:.1f}%  ({churn_counts['Yes']:,} de {len(df):,} clientes)")


## 3.3 Contagem de Evasão por Variáveis Categóricas

### 3.3.1 Perfil Demográfico

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

demo = [
    ('gender',            'Gênero',          ['#3498db', '#e74c3c']),
    ('SeniorCitizen_label','Idoso (≥65 anos)',['#2ecc71', '#e74c3c']),
    ('Churn',             None,              None),   # placeholder
]

configs = [
    ('gender',             'Gênero',           ['#3498db','#e67e22']),
    ('SeniorCitizen_label','Idoso (≥65 anos)', ['#2ecc71','#e74c3c']),
    ('Partner',            'Possui Parceiro(a)',['#9b59b6','#f39c12']),
]

for ax, (col, title, clrs) in zip(axes, configs):
    # Reconstrói valores originais para Partner (já foi mapeado para 0/1)
    if col == 'Partner':
        ct = df.groupby(col)['Churn_num'].mean().reset_index()
        ct[col] = ct[col].map({1: 'Sim', 0: 'Não'})
    else:
        ct = df.groupby(col)['Churn_num'].mean().reset_index()
    ct['Churn_num'] *= 100
    bars = ax.bar(ct[col], ct['Churn_num'], color=clrs, edgecolor='white', width=0.5)
    ax.set_title(f'Taxa de Churn por {title}', fontweight='bold')
    ax.set_ylabel('Taxa de Churn (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 50)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                f'{h:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Churn por Perfil Demográfico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 3.3.2 Tipo de Contrato e Forma de Pagamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Contrato
ct_contract = df.groupby('Contract')['Churn_num'].mean().sort_values(ascending=False) * 100
clrs_c = ['#e74c3c', '#f39c12', '#2ecc71']
bars = axes[0].bar(ct_contract.index, ct_contract.values,
                   color=clrs_c, edgecolor='white', width=0.5)
axes[0].set_title('Taxa de Churn por Tipo de Contrato', fontweight='bold')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_ylim(0, 55)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.5,
                 f'{h:.1f}%', ha='center', fontweight='bold')

# Pagamento
ct_pay = df.groupby('PaymentMethod')['Churn_num'].mean().sort_values(ascending=False) * 100
clrs_p = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
bars2 = axes[1].barh(ct_pay.index, ct_pay.values, color=clrs_p, edgecolor='white')
axes[1].set_title('Taxa de Churn por Forma de Pagamento', fontweight='bold')
axes[1].set_xlabel('Taxa de Churn (%)')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_xlim(0, 55)
for bar in bars2:
    w = bar.get_width()
    axes[1].text(w + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{w:.1f}%', va='center', fontweight='bold')

plt.suptitle('Contrato e Pagamento vs Evasão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 3.3.3 Serviço de Internet

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ct_inet = df.groupby('InternetService')['Churn_num'].mean().sort_values(ascending=False) * 100
clrs_i = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax.bar(ct_inet.index, ct_inet.values, color=clrs_i, edgecolor='white', width=0.5)
ax.set_title('Taxa de Churn por Tipo de Serviço de Internet', fontweight='bold')
ax.set_ylabel('Taxa de Churn (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 55)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
            f'{h:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### 3.3.4 Tempo de Contrato (Tenure)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df[df['Churn']=='No']['tenure'], bins=30,
             alpha=0.7, color='#2ecc71', label='Não Evadiu', edgecolor='white')
axes[0].hist(df[df['Churn']=='Yes']['tenure'], bins=30,
             alpha=0.7, color='#e74c3c', label='Evadiu', edgecolor='white')
axes[0].set_title('Distribuição do Tempo de Contrato', fontweight='bold')
axes[0].set_xlabel('Meses de Contrato')
axes[0].set_ylabel('Quantidade de Clientes')
axes[0].legend()

# Boxplot
bp_data = [df[df['Churn']=='No']['tenure'], df[df['Churn']=='Yes']['tenure']]
bp = axes[1].boxplot(
    bp_data, labels=['Não Evadiu', 'Evadiu'], patch_artist=True,
    medianprops=dict(color='black', linewidth=2.5)
)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[1].set_title('Boxplot — Tenure por Status de Churn', fontweight='bold')
axes[1].set_ylabel('Meses de Contrato')

plt.suptitle('Tempo de Contrato (Tenure) vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mediana tenure — Não Evadiu : {df[df['Churn']=='No']['tenure'].median():.0f} meses")
print(f"Mediana tenure — Evadiu     : {df[df['Churn']=='Yes']['tenure'].median():.0f} meses")


### 3.3.5 Faixa de Cobrança Mensal

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ct_faixa = df.groupby('Faixa_Mensal', observed=True)['Churn_num'].mean() * 100
clrs_f = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
bars = ax.bar(ct_faixa.index, ct_faixa.values, color=clrs_f, edgecolor='white', width=0.5)
ax.set_title('Taxa de Churn por Faixa de Cobrança Mensal', fontweight='bold')
ax.set_ylabel('Taxa de Churn (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 55)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
            f'{h:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### 3.3.6 Serviços Adicionais vs Churn

In [ ]:
services = {
    'OnlineSecurity':   'Segurança Online',
    'OnlineBackup':     'Backup Online',
    'DeviceProtection': 'Proteção Dispositivo',
    'TechSupport':      'Suporte Técnico',
    'StreamingTV':      'Streaming TV',
    'StreamingMovies':  'Streaming Filmes'
}

churn_with    = [df[df[col] == 1]['Churn_num'].mean() * 100 for col in services]
churn_without = [df[df[col] == 0]['Churn_num'].mean() * 100 for col in services]
labels = list(services.values())

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - w/2, churn_without, w, label='Sem o serviço', color='#e74c3c', edgecolor='white')
bars2 = ax.bar(x + w/2, churn_with,    w, label='Com o serviço',  color='#2ecc71', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_title('Taxa de Churn: Com vs Sem Serviço Adicional', fontweight='bold')
ax.set_ylabel('Taxa de Churn (%)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
ax.set_ylim(0, 55)
plt.tight_layout()
plt.show()


---
# 🔗 Seção 4 — Análise de Correlação entre Variáveis (Extra)

## 4.1 Matriz de Correlação — Heatmap

In [ ]:
# Selecionar colunas numéricas relevantes para correlação
num_cols = [
    'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'PaperlessBilling', 'Charges_Monthly', 'Charges_Total',
    'Contas_Diarias', 'Churn_num'
]

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(15, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, annot_kws={'size': 8.5},
    vmin=-1, vmax=1
)
ax.set_title('Matriz de Correlação entre Variáveis — TelecomX',
             fontweight='bold', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


## 4.2 Correlação com Churn — Ranking

In [ ]:
# Ranking das variáveis mais correlacionadas com Churn
corr_churn = corr_matrix['Churn_num'].drop('Churn_num').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
cores = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_churn.values]
bars = ax.barh(corr_churn.index, corr_churn.values, color=cores, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Correlação de Cada Variável com Churn', fontweight='bold')
ax.set_xlabel('Coeficiente de Correlação (Pearson)')
for bar in bars:
    w = bar.get_width()
    ax.text(w + (0.005 if w >= 0 else -0.005),
            bar.get_y() + bar.get_height()/2,
            f'{w:.3f}', va='center',
            ha='left' if w >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

print("\nVariáveis com maior correlação POSITIVA com Churn (fatores de risco):")
print(corr_churn[corr_churn > 0].head(5).to_string())
print("\nVariáveis com maior correlação NEGATIVA com Churn (fatores de proteção):")
print(corr_churn[corr_churn < 0].head(5).to_string())


## 4.3 Conta Diária × Churn (Dispersão)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Tenure × Contas_Diarias colorido por Churn
colors_map = df['Churn_num'].map({0: '#2ecc71', 1: '#e74c3c'})
axes[0].scatter(df['tenure'], df['Contas_Diarias'],
                c=colors_map, alpha=0.3, s=15, edgecolors='none')
axes[0].set_xlabel('Meses de Contrato (Tenure)')
axes[0].set_ylabel('Conta Diária (R$)')
axes[0].set_title('Tenure × Conta Diária por Status de Churn', fontweight='bold')
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=8, label='Não Evadiu'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='Evadiu')
]
axes[0].legend(handles=legend_elements)

# Boxplot: Contas_Diarias por Churn
bp_data2 = [df[df['Churn']=='No']['Contas_Diarias'],
            df[df['Churn']=='Yes']['Contas_Diarias']]
bp2 = axes[1].boxplot(bp_data2, labels=['Não Evadiu', 'Evadiu'],
                      patch_artist=True,
                      medianprops=dict(color='black', linewidth=2.5))
bp2['boxes'][0].set_facecolor('#2ecc71')
bp2['boxes'][1].set_facecolor('#e74c3c')
axes[1].set_title('Boxplot — Conta Diária por Churn', fontweight='bold')
axes[1].set_ylabel('Conta Diária (R$)')

plt.suptitle('Relação entre Conta Diária e Evasão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Média Conta Diária — Não Evadiu : R$ {df[df['Churn']=='No']['Contas_Diarias'].mean():.2f}")
print(f"Média Conta Diária — Evadiu     : R$ {df[df['Churn']=='Yes']['Contas_Diarias'].mean():.2f}")


## 4.4 Quantidade de Serviços Contratados × Churn

In [ ]:
# Número total de serviços adicionais por cliente
servicos = ['OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies']
df['Qtd_Servicos'] = df[servicos].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras: taxa de churn por qtd de serviços
ct_svc = df.groupby('Qtd_Servicos')['Churn_num'].mean() * 100
axes[0].plot(ct_svc.index, ct_svc.values, marker='o', color='#e74c3c', linewidth=2.5, markersize=8)
axes[0].fill_between(ct_svc.index, ct_svc.values, alpha=0.15, color='#e74c3c')
axes[0].set_title('Taxa de Churn por Qtd. de Serviços', fontweight='bold')
axes[0].set_xlabel('Número de Serviços Adicionais')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for x_val, y_val in zip(ct_svc.index, ct_svc.values):
    axes[0].annotate(f'{y_val:.1f}%', (x_val, y_val),
                     textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)

# Distribuição de qtd de serviços por grupo de churn
df_no  = df[df['Churn']=='No']['Qtd_Servicos'].value_counts().sort_index()
df_yes = df[df['Churn']=='Yes']['Qtd_Servicos'].value_counts().sort_index()
x = np.arange(7); w = 0.35
axes[1].bar(x - w/2, [df_no.get(i,0)  for i in range(7)], w,
            label='Não Evadiu', color='#2ecc71', edgecolor='white')
axes[1].bar(x + w/2, [df_yes.get(i,0) for i in range(7)], w,
            label='Evadiu',     color='#e74c3c', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_title('Distribuição de Serviços por Status de Churn', fontweight='bold')
axes[1].set_xlabel('Número de Serviços Adicionais')
axes[1].set_ylabel('Quantidade de Clientes')
axes[1].legend()

plt.suptitle('Quantidade de Serviços vs Evasão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
# 📄 Seção 5 — Relatório Final

## 5.1 Introdução

### Objetivo da Análise
Este projeto tem como objetivo realizar um processo completo de **ETL** (Extração, Transformação e Carga) sobre os dados de clientes da **TelecomX**, seguido de uma **Análise Exploratória de Dados (EDA)** para identificar os principais fatores associados à evasão de clientes — fenômeno conhecido como *Churn*.

### O Problema de Churn
A evasão de clientes é um dos maiores desafios para empresas de telecomunicações. Cada cliente perdido representa não apenas perda de receita recorrente, mas também o alto custo de adquirir um novo cliente para repor a base. Compreender *por que* os clientes cancelam é o primeiro passo para criar estratégias de retenção eficazes e alimentar modelos preditivos de machine learning.

---

## 5.2 Limpeza e Tratamento dos Dados

### Extração
- Dados obtidos diretamente da API (GitHub) em formato **JSON aninhado** com 7.267 registros
- Estrutura normalizada ("flattened") para um DataFrame tabular com **21 colunas**

### Problemas Encontrados e Correções Aplicadas

| Problema | Quantidade | Solução |
|---|---|---|
| Registros com `Churn` em branco | 224 | Removidos (sem rótulo válido) |
| `Charges_Total` como string | 7.267 | Convertido para `float64` |
| `Charges_Total` nulo | 11 | Estimado como `Charges_Monthly × tenure` |
| Colunas binárias (Yes/No) | 11 colunas | Mapeadas para 1/0 |

### Novas Colunas Criadas
- `Churn_num` — versão numérica (0/1) da variável alvo
- `SeniorCitizen_label` — versão legível de SeniorCitizen
- `Contas_Diarias` — cobrança mensal ÷ 30
- `Faixa_Mensal` — categorização do valor mensal em 4 faixas
- `Qtd_Servicos` — soma dos serviços adicionais contratados

**Dataset final:** 7.043 registros × 25 colunas — sem valores nulos.

---

## 5.3 Análise Exploratória de Dados

### Visão Geral


In [ ]:
# Sumário Executivo
total     = len(df)
evadidos  = int(df['Churn_num'].sum())
taxa      = df['Churn_num'].mean() * 100

print("=" * 60)
print("         SUMÁRIO EXECUTIVO — TELECOMX CHURN")
print("=" * 60)
print(f"  Total de clientes analisados  : {total:,}")
print(f"  Clientes que NÃO evadiram     : {total - evadidos:,}  ({100-taxa:.1f}%)")
print(f"  Clientes que EVADIRAM         : {evadidos:,}  ({taxa:.1f}%)")
print()
print("  PRINCIPAIS FATORES DE RISCO IDENTIFICADOS:")
print(f"  ● Contrato mês a mês          : {df[df['Contract']=='Month-to-month']['Churn_num'].mean()*100:.1f}% de churn")
print(f"  ● Pagamento cheque eletrônico : {df[df['PaymentMethod']=='Electronic check']['Churn_num'].mean()*100:.1f}% de churn")
print(f"  ● Internet fibra óptica       : {df[df['InternetService']=='Fiber optic']['Churn_num'].mean()*100:.1f}% de churn")
print(f"  ● Sem segurança online        : {df[df['OnlineSecurity']==0]['Churn_num'].mean()*100:.1f}% de churn")
print(f"  ● Clientes idosos (≥65 anos)  : {df[df['SeniorCitizen']==1]['Churn_num'].mean()*100:.1f}% de churn")
print()
print("  INDICADORES DE PROTEÇÃO CONTRA CHURN:")
print(f"  ✔ Contrato bianual (2 years)  : {df[df['Contract']=='Two year']['Churn_num'].mean()*100:.1f}% de churn")
print(f"  ✔ Tenure mediano (retidos)    : {df[df['Churn']=='No']['tenure'].median():.0f} meses")
print(f"  ✔ Tenure mediano (evadidos)   : {df[df['Churn']=='Yes']['tenure'].median():.0f} meses")
print(f"  ✔ Com 5+ serviços adicionais  : {df[df['Qtd_Servicos']>=5]['Churn_num'].mean()*100:.1f}% de churn")
print("=" * 60)


---

## 5.4 Conclusões e Insights

### 1. Taxa de Evasão Elevada
A TelecomX apresenta uma taxa de churn de **~26,5%** — cerca de 1 em cada 4 clientes cancela o serviço, o que é crítico para a sustentabilidade do negócio.

### 2. Tipo de Contrato é o Principal Fator
Clientes com contratos **mês a mês** têm churn acima de **42%**, enquanto contratos bianuais ficam abaixo de **3%**. A ausência de compromisso de longo prazo é o maior preditor de cancelamento.

### 3. Forma de Pagamento Indica Engajamento
Clientes que pagam por **cheque eletrônico** têm a maior taxa de churn (~45%). Esse perfil pode indicar menor engajamento com a empresa ou maior facilidade psicológica de cancelar.

### 4. Fibra Óptica: Alto Custo, Alta Evasão
Clientes de **fibra óptica** têm churn de ~42%, o maior entre os tipos de internet. Isso sugere possível insatisfação com o custo-benefício ou qualidade percebida do serviço.

### 5. Os Primeiros Meses São Decisivos
A mediana de tenure dos clientes evadidos é de apenas **10 meses**, contra **38 meses** dos que permanecem. O período crítico de retenção é nos primeiros 12 meses de contrato.

### 6. Serviços Adicionais Funcionam como Âncora
Clientes **sem segurança online, suporte técnico ou backup** têm taxas de churn significativamente mais altas. Cada serviço adicional contratado reduz a probabilidade de cancelamento.

### 7. Correlações Relevantes
- **Positiva com Churn**: `Charges_Monthly`, `Contas_Diarias`, `PaperlessBilling`, `SeniorCitizen`
- **Negativa com Churn**: `tenure`, `OnlineSecurity`, `TechSupport`, `Contract` longo

---

## 5.5 Recomendações

| # | Recomendação | Impacto Esperado |
|---|---|---|
| 1 | **Incentivos para contratos longos** — oferecer descontos progressivos para migração de mensal para anual/bianual nos primeiros 6 meses | ⬇️ Redução imediata no churn |
| 2 | **Onboarding proativo** — programa de acompanhamento nos primeiros 12 meses (ligações, tutoriais, benefícios exclusivos) | ⬇️ Churn nos novos clientes |
| 3 | **Incluir segurança online nos pacotes de entrada** — aumenta a âncora de retenção sem custo perceptível ao cliente | ⬇️ Churn em fibra óptica |
| 4 | **Estimular débito automático** como padrão de pagamento, reduzindo o "custo zero" de cancelamento | ⬇️ Churn por cheque eletrônico |
| 5 | **Revisar custo-benefício da fibra óptica** — investigar qualidade, SLA e percepção de valor vs. preço cobrado | ⬇️ Churn em fibra óptica |
| 6 | **Modelo preditivo de risco** com variáveis: `Contract`, `PaymentMethod`, `InternetService`, `tenure`, `OnlineSecurity`, `Charges_Monthly`, `Qtd_Servicos` | 🎯 Ação preventiva personalizada |

---

*📊 Análise produzida como parte do projeto Churn de Clientes — TelecomX Data Science Team.*  
*🐍 Python · Pandas · Matplotlib · Seaborn · NumPy*
